# DBKT - Chương 1. Tổng Quan
## Thu Thập Và Chuẩn Hóa Dữ Liệu Từ FRED Cho Chuỗi Thời Gian Trên R

### 1. Kết nối FRED trong R
Nạp các thư viện R cần thiết (`fredr`, `ggplot2`, `readr`, `dplyr`) và thiết lập FRED API key.

In [ ]:
# 1. Ket noi FRED trong R
# install.packages("fredr")
# install.packages("ggplot2")
# install.packages("readr")
# install.packages("dplyr")

library(fredr)
library(ggplot2)
library(readr)
library(dplyr)

# Cai dat API key
fredr_set_key("b714d7951692a0472a8b387b99497510")

### 2. Lấy dữ liệu GDP với R
Truy xuất chuỗi dữ liệu GDP từ St. Louis Fed (FRED).

In [ ]:
# 2. Lay du lieu GDP voi R
gdp <- fredr(series_id = "GDP")

### 3. Khám phá cấu trúc dữ liệu
Kiểm tra cấu trúc, kích thước và danh sách các biến trong dataset.

In [ ]:
# 3. Kham pha cau truc du lieu
# Kiem tra du lieu
head(gdp)

# Xem cau truc du lieu
str(gdp)

# Kiem tra kich thuoc
dim(gdp)

# Danh sach bien
names(gdp)

### 4. Kiểm tra phạm vi thời gian
Xác định mốc thời gian bắt đầu, mốc kết thúc và khoảng thời gian của chuỗi dữ liệu.

In [ ]:
# 4. Kiem tra pham vi thoi gian
# Ngay dau tien
min(gdp$date)

# Ngay cuoi cung
max(gdp$date)

# Khoang thoi gian
range(gdp$date)

### 5. Đổi tên biến & 6. Sắp xếp theo thời gian
Đổi tên biến `value` thành `gdp` và sắp xếp thứ tự dữ liệu tăng dần theo thời gian.

In [ ]:
# 5. Doi ten bien de de su dung hon
gdp <- gdp |> rename(gdp = value)

# 6. Sap xep theo thoi gian
gdp <- gdp |> arrange(date)

# Hien thi ket qua sau khi chuan hoa
head(gdp)

### 7. Xử lý dữ liệu khuyết (Missing Values)
Kiểm tra tổng quát, đếm số lượng NA, trực quan hóa và thực hiện xử lý dữ liệu khuyết.

In [ ]:
# 7.1 Kiểm tra tổng quát dữ liệu và missing values
str(gdp)
summary(gdp)

# 7.2 Đếm số lượng Missing Values trên toàn bộ bảng dữ liệu
sum(is.na(gdp))

# 7.3 Dem Missing Values theo từng biến
colSums(is.na(gdp))

# 7.4 Trực quan hóa Missing Values
# install.packages("naniar")
library(naniar)
vis_miss(gdp)

In [ ]:
# 7.5 Các phương pháp xử lý Missing Values (Minh họa các cách)

# --- Cách 1: Xóa Missing Values ---
# gdp_clean <- na.omit(gdp)
# Hoặc: gdp_clean <- gdp %>% drop_na()
# (Nên dùng khi số lượng NA rất nhỏ, missing ngẫu nhiên, không ảnh hưởng cấu trúc chuỗi)

# --- Cách 2: Mean Imputation (Điền bằng giá trị trung bình) ---
# gdp$gdp[is.na(gdp$gdp)] <- mean(gdp$gdp, na.rm = TRUE)
# (Lưu ý: Không khuyến nghị cho chuỗi thời gian do làm giảm phương sai và mất seasonal pattern)

# --- Cách 3: LOCF (Last Observation Carried Forward) ---
library(zoo)
# gdp$gdp <- na.locf(gdp$gdp)

# --- Cách 4: Linear Interpolation (Nội suy tuyến tính - Khuyên dùng) ---
gdp$gdp <- na.approx(gdp$gdp)

# 7.6 Kiểm tra lại dữ liệu sau khi xử lý missing values
sum(is.na(gdp))
summary(gdp)

### 8. Khai báo dữ liệu thời gian (Time Series Object)
Tạo đối tượng chuỗi thời gian `ts` với tần suất 4 quý/năm từ Q1-1947.

In [ ]:
# 8. Khai báo dữ liệu thời gian (Time Series Object)
gdp_ts <- ts(gdp$gdp, start = c(1947, 1), frequency = 4)
summary(gdp_ts)